---
# 8. Overall Model Evaluation

In this section, we will be comparing all the CNN Models trained for both the 23px by 23px and 101px by 101px data. This will allow us to choose a CNN Model for submission and also to determine and discuss why a specific CNN is capable of achieveing a highest Test Accuracy than other CNN Models.

---
## 8.1 Model Test Accuracy Comparison

In this section, we will produce the Test Accuracy of all the CNN Models and input them into a DataFrame and sort by the Test Accuracy. Through this we are able to see the best performing CNN Models for each of the input size and create hypothesis and potential explaination on why certain input size or data configuration works better than others.

In [ ]:
# ======================== Evaluating All Models ======================== #
models_dir = "/content/drive/MyDrive/Colab Notebooks/DELE CA1/Models"
results = []

for fname in sorted(os.listdir(models_dir)):
    # ----- Selecting Saved Files ----- #
    if not (fname.endswith(".keras") or fname.endswith(".h5")):
        continue

    model_path = os.path.join(models_dir, fname)
    model = load_model(model_path)

    # ----- Determining Input Size ----- #
    _, h, w, c = model.input_shape
    if (h, w) == (23, 23):
        test_gen = test_generator_23
        size_label = "23×23"
    elif (h, w) == (101, 101):
        test_gen = test_generator_101
        size_label = "101×101"
    else:
        raise ValueError(f"Unexpected input shape {model.input_shape} for {fname}")

    # ----- Evaluating on Appropriate Test Set ----- #
    loss, acc = model.evaluate(test_gen, verbose=0)

    results.append({
        "model_file": fname,
        "input_size": size_label,
        "test_accuracy": acc,
        "test_loss": loss
    })

# ======================== Creating DataFrame ======================== #
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("test_accuracy", ascending=False).reset_index(drop=True)

# ======================== Displaying DataFrame ======================== #
df_results.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to note the following hypothesis.

- The best performing model is the Function API with Data Input of 101px by 101px (No Augmentation)

- The best performing model for 23px by 23px is Resnet-like Model (No Augmentation)

- One reason for why 101px by 101px Data Input performs better for Model Training could be because Larger inputs capture more texture, shape, and fine-grained features, allowing for learning of more complex patterns.

- Another reason for why 101px by 101px Data Input performs better is that it allows for deeper receptive fields for certain models.

- One reason for why Models perform better without Augmentation is becuase, without Augmentation, the data is more stable which can allow for Models to generalise better.

- Over-augmentation can also cause more harm than good in a small dataset as it can distort the features.

With the observation and potential hypothesis discussed above, we will now proceed to store the Best Model and perform analysis on it.

---
## 8.2 Functional API Summary (Best Model)

In this sub-section, we will be producing the Summary of the Functional API Model so as to understand the complexity and layers of the Model. This will allow us to view the various Layers and the various Weights of each layer.

In [ ]:
# ======================== Identifying & Load the Best Model ======================== #
best_idx = df_results['test_accuracy'].idxmax()
best_model_file = df_results.loc[best_idx, 'model_file']
models_dir = "/content/drive/MyDrive/Colab Notebooks/DELE CA1/Models"
best_model_path = os.path.join(models_dir, best_model_file)
best_model = load_model(best_model_path)

# ======================== 2. Model Summary ======================== #
best_model.summary()

With reference to the output above, we are able to view that there are a total of 8 Trainable Layers and 9 Non-Trainable Layers. The total amount of Layers is 17 Layers. The weights for the various layers have been saved and will be submitted.

---
## 8.3 Functional API Architecture (Best Model)

In this sub-section, we will be visualising the various layers of the Functional API Model. The visualisation will show various information such as the Input Shape and Output Shape. This will allow for a better understanding of our Model's Architecture as compared to the Summary as the Visualisation is more streamlined and only displays the important information.

In [ ]:
# ======================== Plotting Architecture ======================== #
plot_model(
    best_model,
    to_file='best_model_architecture.png',
    show_shapes=True,
    show_layer_names=True
)

# ======================== Displaying Architecture ======================== #
display(Image('best_model_architecture.png'))

With reference to the output above, we are able to see the Model's Architecture and the various Input and Output Shape. This allows us to view the Model's input data and also potentially plan for future development should we have more Data or need to increase the complexity of the Functional API Model.

---
## 8.4 Best Model's 3D t-SNE Visualisation

In this sub-section, we will be visualising the Best Model's high-dimesional feature space learnt by the model right before the final classification layer. We will be using t-SNE for visualisation due to the high dimensions and subsequently plot on a 3D Scatter Plot. The potential observations that can be made are listed below.

**Well-separated Clusters**
- The model has learned distinct embeddings for different vegetable classes.
- High inter-class separability (better generalisation and classification confidence).

**Overlapping Clusters**
- Visual similarity between classes
- Confusion likely in the model predictions.
- Potential benefit from more data, augmentation, or deeper networks.

With the potential obervations listed and its meaning, we will proceed to conduct the visualisation in the Code Cell below.

In [ ]:
# ======================== Extracting Penultimate-layer Features ======================== #
penultimate_layer = best_model.layers[-2].output
feat_extractor = Model(inputs=best_model.input, outputs=penultimate_layer)

# ======================== Choosing Appropriate Test Generator ======================== #
_, h, w, _ = best_model.input_shape
if (h, w) == (23, 23):
    test_gen = test_generator_23
elif (h, w) == (101, 101):
    test_gen = test_generator_101
else:
    raise ValueError("Unsupported input size.")

# ======================== Getting Features and True Labels ======================== #
features = feat_extractor.predict(test_gen, verbose=0)
y_true = test_gen.classes
labels = [class_names[i] for i in y_true]

# ======================== t-SNE Projection (3D) ======================== #
tsne = TSNE(n_components=3, perplexity=30, learning_rate='auto', init='pca', random_state=42)
proj = tsne.fit_transform(features)

# ======================== Creating DataFrame ======================== #
df = pd.DataFrame(proj, columns=["X", "Y", "Z"])
df['Label'] = labels

# ======================== Plotting ======================== #
fig = px.scatter_3d(
    df, x='X', y='Y', z='Z',
    color='Label',
    opacity=0.75,
    title='3D t-SNE Projection of Penultimate Layer Embeddings',
    hover_data={'X':False, 'Y':False, 'Z':False, 'Label':True}
)

# ======================== Customising layout ======================== #
fig.update_traces(marker=dict(size=4))
fig.update_layout(
    width=1700, height=1000,
    legend_title_text='Vegetable Class',
    scene=dict(
        xaxis=dict(title='t-SNE X'),
        yaxis=dict(title='t-SNE Y'),
        zaxis=dict(title='t-SNE Z'),
    )
)

# ======================== Displaying Plot ======================== #
fig.show()

With reference to the output of the t-SNE Visualisation above, we are able to observe the various clusters are **Well-Separated** which implies that the Model did classify accuractely and to a high standard. However, there are a few outliers within the plot but these outliers are minor and do not impose a heavy deduction in accuracy towards the model.

---
## 8.5 Visualising Conv2D Layers' Channels

In this sub-section, we will be visualising the various Conv2D Layers' Channels which can give us insights on what the Best Model 'sees' which can allow for Model Transparency and also view the Layer Hierachy within this Model. It is expected that some Channels be dead due to regularisation, but this should not be a frequent occurance. With the reasoning for this operation and the expectations indicated, we will proceed to perform the visualisation in the Code Cell below.

In [ ]:
# ==================== Finding All Conv2D Layers ==================== #
conv_layer_names = [layer.name for layer in best_model.layers if isinstance(layer, Conv2D)]

# ==================== Building Multi-Layer Output Model ==================== #
activation_outputs = [best_model.get_layer(name).output for name in conv_layer_names]
activation_model = Model(inputs=best_model.input, outputs=activation_outputs)

# ==================== Selecting One Sample Image ==================== #
img_batch, _ = test_gen[0]  # batch of test images
img_input = img_batch[0:1]  # take first image only
activations = activation_model.predict(img_input)

# ==================== Plotting Feature Maps for All Conv Layers ==================== #
for layer_name, feature_map in zip(conv_layer_names, activations):
    num_channels = feature_map.shape[-1]
    n_cols = 6
    n_rows = (num_channels + n_cols - 1) // n_cols

    plt.figure(figsize=(14, 2.2 * n_rows))
    plt.suptitle(f"Activations — {layer_name} ({num_channels} channels)", fontsize=16)

    for i in range(min(num_channels, 64)):
        ax = plt.subplot(n_rows, n_cols, i+1)
        ax.imshow(feature_map[0, :, :, i], cmap='viridis')
        ax.axis('off')
        ax.set_title(f"Ch {i}", fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()

With reference to the output of the Visualisation above, we are able to see that the image presented strong edge-based activations concentrated around several consistent regions, particularly diagonal and curved textures. This suggests the model is extracting robust local patterns such as ridges, leaf veins, or structured skin, which serve as key visual features for distinguishing between similar vegetable classes.

---
## 8.6 Visualising Overall Architecture

In this sub-section, we will be visualising the Overall Architecture of the Best Model. This will allow for a very vague overview of the Model's architecture without much information on weights and inputs or outputs which will be more friendly towards audience with no AI background. This will also allow us to see the Model's process in one view.

In [ ]:
# ==================== Producing 3D‐block View ==================== #
visualkeras.layered_view(
    best_model,
    to_file='model_3d.png',
    legend=True,
    spacing=40,
    scale_xy=1.0,
    scale_z=0.5
)

# ==================== Displaying 3D‐block View ==================== #
Image('model_3d.png')

With reference to the output of the Model's Achitecture above, we are able to see that the Model has multiple layers before the output. We are also able to see the sequence in which the process is taking place.